# Sea Grape Stage-1 Detector — Colab Training

Trains the single-class YOLO11s sea grape detector on 640px tiles.

**Before you start:** set the runtime to a GPU.
`Runtime -> Change runtime type -> Hardware accelerator: T4 GPU`

On a T4 this run takes roughly 20-30 minutes, against ~6.8 hours on Apple MPS.

You need `tiles.zip` (92 MB) from `build/` on your local machine.

## 1. Confirm the GPU is actually attached

If this prints `cpu`, stop and fix the runtime type — otherwise you gain nothing over training locally.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 2. Install Ultralytics

In [ ]:
!pip install -q ultralytics
import ultralytics
print('ultralytics', ultralytics.__version__)

## 3. Get the data in

Two options. **Drive is the better one** — a Colab runtime can disconnect and you would otherwise re-upload 92 MB each time.

### Option A — Google Drive (recommended)
Upload `tiles.zip` to your Drive first, then run the cell below.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Adjust if you put the zip somewhere other than the Drive root.
ZIP_PATH = '/content/drive/MyDrive/tiles.zip'

### Option B — direct upload

Skip this cell if you used Drive. The upload is slow and does not survive a disconnect.

In [ ]:
from google.colab import files
uploaded = files.upload()          # pick tiles.zip
ZIP_PATH = '/content/tiles.zip'

## 4. Unzip and repoint `data.yaml`

The `path:` line in `data.yaml` is an **absolute** path to the Mac that built it. Ultralytics resolves the dataset against that key, so it has to be rewritten for this machine or training silently finds no images.

In [ ]:
import zipfile, os
from pathlib import Path

with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall('/content/data')

ROOT = Path('/content/data/tiles')

# Stale label caches from the source machine would be reused and break paths.
for cache in ROOT.rglob('*.cache'):
    cache.unlink()

(ROOT / 'data.yaml').write_text(
    f'path: {ROOT}\n'
    'train: train/images\n'
    'val: valid/images\n'
    'test: test/images\n'
    '\n'
    'nc: 1\n'
    'names: ["SeaGrape"]\n'
)
print((ROOT / 'data.yaml').read_text())

for split in ['train', 'valid', 'test']:
    n_img = len(list((ROOT / split / 'images').glob('*.jpg')))
    n_lbl = len(list((ROOT / split / 'labels').glob('*.txt')))
    print(f'{split:<6} {n_img:>5} images  {n_lbl:>5} labels')

Expected: **1243 / 425 / 264**. If images and labels disagree, the zip is incomplete — rebuild it locally with `pipeline/02_slice_tiles.py`.

## 5. Train

Hyperparameters match `pipeline/05_train_detector.py` exactly, with two changes for this hardware: `batch` 16 to 32 (the T4 has the memory) and `workers` 8 to 2 (Colab gives 2 vCPUs).

The augmentation choices are the load-bearing part — see the comments.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11s.pt')

model.train(
    data=str(ROOT / 'data.yaml'),
    epochs=120,
    patience=30,
    imgsz=640,
    batch=32,          # T4 has the memory; drop to 16 if you hit OOM
    device=0,
    workers=2,         # Colab gives 2 vCPUs
    project='/content/runs',
    name='stage1_yolo11s',
    exist_ok=True,

    # Colour jitter is safe here: stage 1 is single-class, so shifting hue
    # cannot flip a maturity label. The stage-2 classifier is the opposite
    # case, where the class IS the colour and this would relabel the data.
    hsv_h=0.015,
    hsv_s=0.4,
    hsv_v=0.4,

    # Grapes have no canonical orientation, so rotation and both flips are
    # free extra data.
    degrees=90.0,
    fliplr=0.5,
    flipud=0.5,

    # Small translate keeps tiny objects from being shoved off the tile.
    translate=0.1,
    scale=0.3,
    shear=0.0,
    perspective=0.0,

    mosaic=0.5,
    close_mosaic=15,   # last 15 epochs tune box regression on clean tiles
    mixup=0.0,         # would overlay ghost grapes onto real ones
    erasing=0.0,       # would delete grapes without deleting their labels

    cos_lr=True,
    optimizer='AdamW',
    lr0=1e-3,
    plots=True,
)

## 6. Read the results

**Recall is the metric to trust here.** The annotators labelled only a subset of the resolvable grapes, so a correctly-found unlabelled grape scores as a false positive — precision and mAP are floors, not estimates. A missed grape is gone from the count for good, while a false positive still gets rejected by the stage-2 classifier.

In [ ]:
import pandas as pd

df = pd.read_csv('/content/runs/stage1_yolo11s/results.csv')
df.columns = [c.strip() for c in df.columns]
best = df.loc[df['metrics/mAP50(B)'].idxmax()]

print(f"Best epoch : {int(best['epoch'])}")
print(f"Recall     : {best['metrics/recall(B)']:.4f}   <- trust this")
print(f"Precision  : {best['metrics/precision(B)']:.4f}   <- lower bound only")
print(f"mAP50      : {best['metrics/mAP50(B)']:.4f}")
print(f"mAP50-95   : {best['metrics/mAP50-95(B)']:.4f}")

df[['epoch', 'metrics/precision(B)', 'metrics/recall(B)',
    'metrics/mAP50(B)', 'metrics/mAP50-95(B)']].tail(10)

In [ ]:
from IPython.display import Image, display
display(Image('/content/runs/stage1_yolo11s/results.png', width=1000))

## 7. Evaluate on the held-out test tiles

In [ ]:
best_model = YOLO('/content/runs/stage1_yolo11s/weights/best.pt')
metrics = best_model.val(data=str(ROOT / 'data.yaml'), split='test', device=0, max_det=1000)
print(f'\nTest recall    : {metrics.box.r[0]:.4f}')
print(f'Test precision : {metrics.box.p[0]:.4f}')
print(f'Test mAP50     : {metrics.box.map50:.4f}')

## 8. Get the weights back out

Do this before the runtime disconnects. Drop `best.pt` into `build/runs/stage1_yolo11s/weights/` locally and `pipeline/predict.py` will pick it up with no changes.

In [ ]:
# Straight download
from google.colab import files
files.download('/content/runs/stage1_yolo11s/weights/best.pt')

In [ ]:
# Or copy the whole run to Drive, which also keeps the plots and CSV
!mkdir -p /content/drive/MyDrive/seagrape_runs
!cp -r /content/runs/stage1_yolo11s /content/drive/MyDrive/seagrape_runs/
print('Copied to Drive: seagrape_runs/stage1_yolo11s')

---
## Troubleshooting

| Symptom | Cause | Fix |
| --- | --- | --- |
| `train: 0 images` | `data.yaml` still points at the Mac path | Rerun cell 4 |
| Labels look wrong / stale | A `.cache` file survived the unzip | Cell 4 deletes them; rerun it |
| CUDA out of memory | `batch=32` too large | Set `batch=16` |
| Very slow, ~200s/epoch | Running on CPU | Fix runtime type, rerun from cell 1 |
| Runtime disconnected mid-run | Colab idle timeout | Reload, rerun 1-4, then `YOLO('.../last.pt').train(resume=True)` |

## Training the stage-2 classifier too

Same pattern with `crops.zip` (24 MB) and `pipeline/06_train_classifier.py`. It only takes ~20 minutes on MPS, so it is rarely worth moving — the detector is the bottleneck.